# **Environment and Path Setup**

## **Dependency Verification & Installation**

In [1]:
import importlib
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
    "mlxtend": "mlxtend",
}

for module_name, pip_name in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        print(f"Installing {pip_name} into active kernel ({sys.executable})...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

print("All required packages are available in the active kernel.")

All required packages are available in the active kernel.


Checks whether pandas, numpy, matplotlib, seaborn, scikit-learn, and lmxtend are already installed in the active Jupyter Kernet and installs any missing packages via pip so the notebook runs reliably in both VS Code and Google Colab. 

## **Runtime Environment Detection & Repository Setup**

In [2]:
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
GITHUB_USER = "ShaunCipher"
REPO_NAME = "Jusgado_data-mining-capstone"
BRANCH = "main"

if IN_COLAB:
    repo_dir = Path(f"/content/{REPO_NAME}")
    if not repo_dir.exists():
        print(f"Cloning {GITHUB_USER}/{REPO_NAME} ({BRANCH}) for Google Colab...")
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "-b",
                BRANCH,
                f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git",
                str(repo_dir),
            ],
            check=True,
        )
    PROJECT_ROOT = repo_dir
else:
    cwd = Path.cwd()
    if (cwd / "datasets").exists():
        PROJECT_ROOT = cwd
    elif (cwd.parent / "datasets").exists():
        PROJECT_ROOT = cwd.parent
    else:
        PROJECT_ROOT = cwd

print("Runtime Environment:", "Google Colab" if IN_COLAB else "Local VS Code")
print("Python Executable:", sys.executable)
print("Project Root:", PROJECT_ROOT.resolve())

Runtime Environment: Local VS Code
Python Executable: c:\Users\Jusgado\AppData\Local\Python\pythoncore-3.14-64\python.exe
Project Root: C:\Users\Jusgado\OneDrive\Documents\Github\Jusgado_data-mining-capstone


Detects whether the notebook is running in Google Colab or locally in VS Code, clones ShaunCipher/Jusgado_data-mining-capstone into /content when executed in Colab, and resolves PROJECT_ROOT consistently across local and cloud runs.

## **Dataset Directory & File Inventory Check**

In [3]:
from pathlib import Path

DATA_DIR = PROJECT_ROOT / "datasets"
assert DATA_DIR.exists(), f"Datasets folder not found at: {DATA_DIR.resolve()}"

discovered_items = sorted(
    [p.relative_to(DATA_DIR) for p in DATA_DIR.rglob("*") if p.is_file()]
)

print("Resolved DATA_DIR:", DATA_DIR.resolve())
print(f"Total files discovered under datasets/: {len(discovered_items)}")
for item in discovered_items:
    print(" -", item)

Resolved DATA_DIR: C:\Users\Jusgado\OneDrive\Documents\Github\Jusgado_data-mining-capstone\datasets
Total files discovered under datasets/: 76
 - menu_items\menu_items.csv
 - payment_methods\payment_methods.csv
 - stores\stores.csv
 - transaction_items\transaction_items_202307.csv
 - transaction_items\transaction_items_202308.csv
 - transaction_items\transaction_items_202309.csv
 - transaction_items\transaction_items_202310.csv
 - transaction_items\transaction_items_202311.csv
 - transaction_items\transaction_items_202312.csv
 - transaction_items\transaction_items_202401.csv
 - transaction_items\transaction_items_202402.csv
 - transaction_items\transaction_items_202403.csv
 - transaction_items\transaction_items_202404.csv
 - transaction_items\transaction_items_202405.csv
 - transaction_items\transaction_items_202406.csv
 - transaction_items\transaction_items_202407.csv
 - transaction_items\transaction_items_202408.csv
 - transaction_items\transaction_items_202409.csv
 - transaction_ite

Verifies that the datasets/ directory exists under PROJECT_ROOT and lists every file found inside it in order to confirm the raw dataset files are in place before loading them into SQLite.

Brief Description of Datasets: 
*  menu_items.csv - Contains the items available at the menu
*  payment_methods.csv - Contains the available payment method 
*  stores.csv - Contains participating stores address
*  transaction_items_yyyymm.csv - Contains the transactions information regarding items from January of 2024 to June of 2025
*  transaction_yyyymm.csv - Contains the transactions information regarding customers from December of 2023 to June of 2025
*  users_yyyymm.csv - Contains the customer information
*  vouchers.csv - Contains available vouchers for transactions

## **Load CSV Datasets into SQLite & Summarize Schemas**

In [4]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
csv_files = sorted(DATA_DIR.rglob("*.csv"))
assert len(csv_files) > 0, f"No CSV files found under {DATA_DIR.resolve()}"

summary_rows = []
loaded_tables = {}

for csv_path in csv_files:
    table_name = csv_path.stem.lower().replace("-", "_").replace(" ", "_")
    df = pd.read_csv(csv_path)
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    loaded_tables[table_name] = df

    summary_rows.append(
        {
            "table_name": table_name,
            "file": str(csv_path.relative_to(DATA_DIR)),
            "rows": len(df),
            "cols": len(df.columns),
            "columns": ", ".join(df.columns),
        }
    )

schema_summary_df = pd.DataFrame(summary_rows)
display(schema_summary_df)

,table_name,file,rows,cols,columns
0,menu_items,menu_items\menu_items.csv,8,7,"item_id, item_name, category, price, is_season..."
1,payment_methods,payment_methods\payment_methods.csv,5,3,"method_id, method_name, category"
2,stores,stores\stores.csv,10,8,"store_id, store_name, street, postal_code, cit..."
3,transaction_items_202307,transaction_items\transaction_items_202307.csv,1241261,6,"transaction_id, item_id, quantity, unit_price,..."
4,transaction_items_202308,transaction_items\transaction_items_202308.csv,1238735,6,"transaction_id, item_id, quantity, unit_price,..."
...,...,...,...,...,...
71,users_202503,users\users_202503.csv,182902,4,"user_id, gender, birthdate, registered_at"
72,users_202504,users\users_202504.csv,178617,4,"user_id, gender, birthdate, registered_at"
73,users_202505,users\users_202505.csv,184015,4,"user_id, gender, birthdate, registered_at"
74,users_202506,users\users_202506.csv,178713,4,"user_id, gender, birthdate, registered_at"


Reads all discovered CSV files into Pandas DataFrames, loads each table into an in-memory SQLite database for SQL querying, stores the DataFrames in loaded_tables, and displays a summary table of row counts, column counts, and column names.